In [4]:

# Day 8 - Step 1: Revenue Data Readiness Audit
# Corrected version: define project_dir first

from pathlib import Path
import pandas as pd

# Define the project root explicitly
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

# Verify the project folder
if not project_dir.exists():
    raise FileNotFoundError(
        f"Project folder not found: {project_dir}"
    )

folders_to_check = [
    project_dir / "data" / "raw",
    project_dir / "data" / "clean",
    project_dir / "data" / "processed",
    project_dir / "data" / "synthetic",
]

revenue_keywords = [
    "revenue",
    "invoice",
    "payment",
    "collection",
    "billed_amount",
    "collected_amount",
    "outstanding_amount",
    "transaction",
    "sales",
]

audit_rows = []

for folder in folders_to_check:
    if not folder.exists():
        print(f"Folder not found, skipping: {folder}")
        continue

    for file_path in sorted(folder.glob("*.csv")):
        try:
            # Read column names and a small sample only
            df = pd.read_csv(file_path, nrows=5)
            columns = list(df.columns)

            matched_columns = [
                col for col in columns
                if any(
                    keyword in col.lower()
                    for keyword in revenue_keywords
                )
            ]

            filename_match = any(
                keyword in file_path.stem.lower()
                for keyword in revenue_keywords
            )

            audit_rows.append({
                "File": file_path.name,
                "Folder": str(
                    file_path.parent.relative_to(project_dir)
                ),
                "Columns_Sampled": len(columns),
                "Revenue_Column_Matches": ", ".join(
                    matched_columns
                ),
                "Filename_Match": filename_match,
                "Potential_Revenue_Source": (
                    bool(matched_columns) or filename_match
                ),
            })

        except Exception as exc:
            audit_rows.append({
                "File": file_path.name,
                "Folder": str(
                    file_path.parent.relative_to(project_dir)
                ),
                "Columns_Sampled": None,
                "Revenue_Column_Matches": "",
                "Filename_Match": False,
                "Potential_Revenue_Source": False,
                "Read_Error": str(exc),
            })

audit_df = pd.DataFrame(audit_rows)

# Save the audit in the processed data folder
output_path = (
    project_dir
    / "data"
    / "processed"
    / "Day8_Revenue_Data_Readiness_Audit.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(output_path, index=False)

print("\nRevenue Data Readiness Audit")
print("-" * 35)

if not audit_df.empty:
    print(audit_df.to_string(index=False))
else:
    print("No CSV files were found in the audited folders.")

print("\nCSV files audited:", len(audit_df))
print(
    "Potential revenue sources identified:",
    int(audit_df["Potential_Revenue_Source"].sum())
    if not audit_df.empty else 0
)
print("Saved to:", output_path)
print("File exists:", output_path.exists())



Revenue Data Readiness Audit
-----------------------------------
                                                       File         Folder  Columns_Sampled                           Revenue_Column_Matches  Filename_Match  Potential_Revenue_Source
                                 NexziByte_Final_Scored.csv       data\raw               25                                                            False                     False
                                   Day3_Prospects_Clean.csv     data\clean               28                                                            False                     False
                    Day10_Churn_Model_Feasibility_Audit.csv data\processed                4                                                            False                     False
                       Day10_CLV_Feasibility_Assessment.csv data\processed                5                                                            False                     False
                   

In [5]:

# Day 8 - Step 2: Inspect Existing Dataset Columns

for folder in folders_to_check:
    if not folder.exists():
        continue

    for file_path in sorted(folder.glob("*.csv")):
        try:
            df = pd.read_csv(file_path, nrows=3)

            print(f"\nFile: {file_path.name}")
            print("Columns:", list(df.columns))

        except Exception as exc:
            print(f"\nCould not inspect {file_path.name}: {exc}")



File: NexziByte_Final_Scored.csv
Columns: ['Company Name', 'Website / Domain', 'Industry / Sub-industry', 'HQ Location (City, State)', 'Founded Year (Standardized)', 'Products/Services', 'Target Customers / B2B or B2C', 'Technology Signals', 'Recent Business Signal', 'Decision-Maker (Name + Title)', 'Source(s) Used', 'Date Verified', 'Employee Count (Numeric)', 'Website Validation', 'Employee Validation', 'Duplicate Flag', 'Data Quality Status', 'Company Size Score', 'B2B Fit Score', 'Technology Signal Score', 'Business Need Score', 'Recent Business Signal Score', 'Decision-Maker Score', 'Prospect Score', 'Prospect Classification']

File: Day3_Prospects_Clean.csv
Columns: ['Company Name', 'Website / Domain', 'Industry / Sub-industry', 'HQ Location (City, State)', 'Founded Year (Standardized)', 'Products/Services', 'Target Customers / B2B or B2C', 'Technology Signals', 'Recent Business Signal', 'Decision-Maker (Name + Title)', 'Source(s) Used', 'Date Verified', 'Employee Count (Numeric

In [6]:

# Day 8 - Step 3: Generate Synthetic Monthly Revenue
# DEMONSTRATION ONLY — NOT ACTUAL NEXZIBYTE REVENUE

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# Generate 36 months of illustrative revenue
dates = pd.date_range(
    start="2023-01-01",
    periods=36,
    freq="MS"
)

month_number = np.arange(36)

# Illustrative trend, seasonality, and random variation
base_revenue = 500000
trend = month_number * 8000
seasonality = 50000 * np.sin(
    2 * np.pi * month_number / 12
)
noise = rng.normal(0, 30000, size=36)

revenue = np.maximum(
    base_revenue + trend + seasonality + noise,
    100000
)

synthetic_revenue = pd.DataFrame({
    "Month": dates,
    "Revenue": np.round(revenue, 2),
    "Synthetic_Data_Flag": True,
    "Data_Use": "Forecasting demonstration only"
})

output_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Synthetic_Monthly_Revenue.csv"
)

synthetic_revenue.to_csv(output_path, index=False)

print("Dataset shape:", synthetic_revenue.shape)
print("\nFirst 5 records:")
print(synthetic_revenue.head().to_string(index=False))

print("\nDate range:",
      synthetic_revenue["Month"].min().date(),
      "to",
      synthetic_revenue["Month"].max().date())

print("Missing values:",
      int(synthetic_revenue.isna().sum().sum()))

print("All records marked synthetic:",
      bool(synthetic_revenue["Synthetic_Data_Flag"].all()))

print("\nSaved to:", output_path)


Dataset shape: (36, 4)

First 5 records:
     Month   Revenue  Synthetic_Data_Flag                       Data_Use
2023-01-01 509141.51                 True Forecasting demonstration only
2023-02-01 501800.48                 True Forecasting demonstration only
2023-03-01 581814.81                 True Forecasting demonstration only
2023-04-01 602216.94                 True Forecasting demonstration only
2023-05-01 516770.21                 True Forecasting demonstration only

Date range: 2023-01-01 to 2025-12-01
Missing values: 0
All records marked synthetic: True

Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Synthetic_Monthly_Revenue.csv


In [7]:

# Day 8 - Step 4: Naive Forecast Baseline
# SYNTHETIC DEMONSTRATION ONLY

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)
import numpy as np
import pandas as pd

# Ensure chronological order
revenue_df = synthetic_revenue.copy()
revenue_df["Month"] = pd.to_datetime(revenue_df["Month"])
revenue_df = revenue_df.sort_values("Month").reset_index(drop=True)

# Hold out the final 6 months for testing
train_df = revenue_df.iloc[:-6].copy()
test_df = revenue_df.iloc[-6:].copy()

# Naive forecast: repeat the latest training observation
last_train_revenue = train_df["Revenue"].iloc[-1]

naive_predictions = np.repeat(
    last_train_revenue,
    len(test_df)
)

mae = mean_absolute_error(
    test_df["Revenue"],
    naive_predictions
)

rmse = np.sqrt(mean_squared_error(
    test_df["Revenue"],
    naive_predictions
))

baseline_results = pd.DataFrame({
    "Month": test_df["Month"],
    "Actual_Revenue": test_df["Revenue"].to_numpy(),
    "Naive_Forecast": naive_predictions
})

metrics_df = pd.DataFrame([{
    "Model": "Naive Forecast",
    "Test_Months": len(test_df),
    "MAE": mae,
    "RMSE": rmse,
    "Data_Status": "Synthetic demonstration only"
}])

forecast_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Naive_Forecast_Backtest.csv"
)

metrics_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Naive_Forecast_Metrics.csv"
)

baseline_results.to_csv(forecast_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print("Training months:", len(train_df))
print("Testing months:", len(test_df))
print("\nBacktest results:")
print(baseline_results.to_string(index=False))

print("\nBaseline metrics:")
print(metrics_df.to_string(index=False))

print("\nSaved forecast to:", forecast_path)
print("Saved metrics to:", metrics_path)


Training months: 30
Testing months: 6

Backtest results:
     Month  Actual_Revenue  Naive_Forecast
2025-07-01       804249.43       769924.63
2025-08-01       710807.55       769924.63
2025-09-01       697331.45       769924.63
2025-10-01       689586.82       769924.63
2025-11-01       747178.11       769924.63
2025-12-01       788869.17       769924.63

Baseline metrics:
         Model  Test_Months       MAE        RMSE                  Data_Status
Naive Forecast            6 48010.655 53655.15556 Synthetic demonstration only

Saved forecast to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Naive_Forecast_Backtest.csv
Saved metrics to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Naive_Forecast_Metrics.csv


In [8]:

# Day 8 - Step 5: Seasonal Naive Forecast
# SYNTHETIC DEMONSTRATION ONLY

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the existing chronological train/test split.
# Each test month is forecast from the same month 12 months earlier.
seasonal_predictions = []

for test_index in test_df.index:
    source_index = test_index - 12
    seasonal_predictions.append(
        revenue_df.loc[source_index, "Revenue"]
    )

seasonal_predictions = np.array(seasonal_predictions)

seasonal_mae = mean_absolute_error(
    test_df["Revenue"], seasonal_predictions
)

seasonal_rmse = np.sqrt(mean_squared_error(
    test_df["Revenue"], seasonal_predictions
))

seasonal_results = pd.DataFrame({
    "Month": test_df["Month"].to_numpy(),
    "Actual_Revenue": test_df["Revenue"].to_numpy(),
    "Seasonal_Naive_Forecast": seasonal_predictions
})

comparison_df = pd.DataFrame([
    {
        "Model": "Naive Forecast",
        "MAE": mean_absolute_error(
            test_df["Revenue"], naive_predictions
        ),
        "RMSE": np.sqrt(mean_squared_error(
            test_df["Revenue"], naive_predictions
        )),
        "Data_Status": "Synthetic demonstration only"
    },
    {
        "Model": "Seasonal Naive Forecast",
        "MAE": seasonal_mae,
        "RMSE": seasonal_rmse,
        "Data_Status": "Synthetic demonstration only"
    }
])

forecast_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Seasonal_Naive_Forecast_Backtest.csv"
)

comparison_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Forecast_Model_Comparison.csv"
)

seasonal_results.to_csv(forecast_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

print("Seasonal forecast results:")
print(seasonal_results.to_string(index=False))

print("\nModel comparison:")
print(comparison_df.to_string(index=False))

print("\nSaved seasonal forecast to:", forecast_path)
print("Saved model comparison to:", comparison_path)


Seasonal forecast results:
     Month  Actual_Revenue  Seasonal_Naive_Forecast
2025-07-01       804249.43                670353.51
2025-08-01       710807.55                625502.22
2025-09-01       697331.45                611152.86
2025-10-01       689586.82                597572.11
2025-11-01       747178.11                669374.97
2025-12-01       788869.17                654364.12

Model comparison:
                  Model           MAE          RMSE                  Data_Status
         Naive Forecast  48010.655000  53655.155560 Synthetic demonstration only
Seasonal Naive Forecast 101617.123333 104278.081765 Synthetic demonstration only

Saved seasonal forecast to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Seasonal_Naive_Forecast_Backtest.csv
Saved model comparison to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Forecast_Model_Comparison.csv


In [9]:

# Day 8 - Step 6: Linear Trend Forecast
# SYNTHETIC DEMONSTRATION ONLY

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use time index as the predictor
X_train_time = np.arange(len(train_df)).reshape(-1, 1)
X_test_time = np.arange(
    len(train_df), len(train_df) + len(test_df)
).reshape(-1, 1)

trend_model = LinearRegression()
trend_model.fit(X_train_time, train_df["Revenue"])

trend_predictions = trend_model.predict(X_test_time)

trend_mae = mean_absolute_error(
    test_df["Revenue"], trend_predictions
)

trend_rmse = np.sqrt(mean_squared_error(
    test_df["Revenue"], trend_predictions
))

trend_results = pd.DataFrame({
    "Month": test_df["Month"].to_numpy(),
    "Actual_Revenue": test_df["Revenue"].to_numpy(),
    "Linear_Trend_Forecast": np.round(trend_predictions, 2)
})

comparison_df = pd.DataFrame([
    {
        "Model": "Naive Forecast",
        "MAE": mean_absolute_error(
            test_df["Revenue"], naive_predictions
        ),
        "RMSE": np.sqrt(mean_squared_error(
            test_df["Revenue"], naive_predictions
        ))
    },
    {
        "Model": "Seasonal Naive Forecast",
        "MAE": seasonal_mae,
        "RMSE": seasonal_rmse
    },
    {
        "Model": "Linear Trend Forecast",
        "MAE": trend_mae,
        "RMSE": trend_rmse
    }
])

comparison_df["Data_Status"] = "Synthetic demonstration only"

trend_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Linear_Trend_Forecast_Backtest.csv"
)

comparison_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Forecast_Model_Comparison.csv"
)

trend_results.to_csv(trend_path, index=False)
comparison_df.to_csv(comparison_path, index=False)

print("Linear trend backtest:")
print(trend_results.to_string(index=False))

print("\nUpdated model comparison:")
print(comparison_df.sort_values("MAE").to_string(index=False))

print("\nSaved trend forecast to:", trend_path)
print("Updated model comparison:", comparison_path)


Linear trend backtest:
     Month  Actual_Revenue  Linear_Trend_Forecast
2025-07-01       804249.43              755091.11
2025-08-01       710807.55              763630.89
2025-09-01       697331.45              772170.67
2025-10-01       689586.82              780710.45
2025-11-01       747178.11              789250.24
2025-12-01       788869.17              797790.02

Updated model comparison:
                  Model           MAE          RMSE                  Data_Status
         Naive Forecast  48010.655000  53655.155560 Synthetic demonstration only
  Linear Trend Forecast  53156.248728  59105.720754 Synthetic demonstration only
Seasonal Naive Forecast 101617.123333 104278.081765 Synthetic demonstration only

Saved trend forecast to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Linear_Trend_Forecast_Backtest.csv
Updated model comparison: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Forecast_Model_Comparison.csv


In [10]:

# Day 8 - Step 7: 3, 6, and 12-Month Revenue Forecast
# SYNTHETIC DEMONSTRATION ONLY — NOT ACTUAL NEXZIBYTE REVENUE

import numpy as np
import pandas as pd

# Load the complete synthetic revenue dataset.
# Do not use train_df here because it only contains training observations.
revenue_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Synthetic_Monthly_Revenue.csv"
)

revenue_df = pd.read_csv(revenue_path, parse_dates=["Month"])
revenue_df = revenue_df.sort_values("Month").reset_index(drop=True)

# Basic data validation
required_columns = {"Month", "Revenue", "Synthetic_Data_Flag", "Data_Use"}

if not required_columns.issubset(revenue_df.columns):
    raise ValueError(
        f"Missing required columns: "
        f"{required_columns - set(revenue_df.columns)}"
    )

if revenue_df.empty:
    raise ValueError("Revenue dataset is empty.")

if revenue_df["Revenue"].isna().any():
    raise ValueError("Revenue dataset contains missing revenue values.")

if not revenue_df["Synthetic_Data_Flag"].astype(bool).all():
    raise ValueError("Expected all records to be flagged as synthetic.")

# Use all observations for the final forward forecast.
last_observed_month = revenue_df["Month"].max()
last_observed_revenue = float(revenue_df["Revenue"].iloc[-1])

training_values = revenue_df["Revenue"].to_numpy(dtype=float)

# Approximate residual spread from historical one-step naive forecasts.
training_residuals = np.diff(training_values)

if len(training_residuals) < 2:
    raise ValueError("Not enough observations to estimate residual spread.")

residual_std = float(np.std(training_residuals, ddof=1))

forecast_rows = []

for horizon in [3, 6, 12]:
    for step in range(1, horizon + 1):
        forecast_month = (
            last_observed_month + pd.DateOffset(months=step)
        )

        # Approximate interval only; not a calibrated prediction interval.
        interval_width = 1.96 * residual_std * np.sqrt(step)

        forecast_rows.append({
            "Month": forecast_month,
            "Horizon_Months": horizon,
            "Step_Ahead": step,
            "Forecast_Revenue": round(last_observed_revenue, 2),
            "Approx_Lower_95": round(
                max(0, last_observed_revenue - interval_width), 2
            ),
            "Approx_Upper_95": round(
                last_observed_revenue + interval_width, 2
            ),
            "Model": "Naive Forecast",
            "Data_Status": "Synthetic demonstration only",
            "Interval_Note": (
                "Approximate residual-based interval; "
                "not formally calibrated or validated"
            )
        })

forecast_df = pd.DataFrame(forecast_rows)

# Save separate files for each forecast horizon.
for horizon in [3, 6, 12]:
    horizon_df = forecast_df[
        forecast_df["Horizon_Months"] == horizon
    ].copy()

    output_path = (
        project_dir / "data" / "synthetic"
        / f"Day8_Synthetic_Revenue_Forecast_{horizon}M.csv"
    )

    horizon_df.to_csv(output_path, index=False)

    print(f"\n{horizon}-month forecast saved to: {output_path}")
    print("Rows:", len(horizon_df))
    print(
        "Forecast period:",
        horizon_df["Month"].min().strftime("%Y-%m"),
        "to",
        horizon_df["Month"].max().strftime("%Y-%m")
    )

print("\nForecast preview:")
print(forecast_df.head(12).to_string(index=False))

print("\nTotal forecast rows across all three files:", len(forecast_df))
print("Latest historical month:", last_observed_month.strftime("%Y-%m"))
print("Latest synthetic revenue:", round(last_observed_revenue, 2))
print("\nStep 7 completed. Results remain synthetic demonstrations.")


3-month forecast saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Synthetic_Revenue_Forecast_3M.csv
Rows: 3
Forecast period: 2026-01 to 2026-03

6-month forecast saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Synthetic_Revenue_Forecast_6M.csv
Rows: 6
Forecast period: 2026-01 to 2026-06

12-month forecast saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Synthetic_Revenue_Forecast_12M.csv
Rows: 12
Forecast period: 2026-01 to 2026-12

Forecast preview:
     Month  Horizon_Months  Step_Ahead  Forecast_Revenue  Approx_Lower_95  Approx_Upper_95          Model                  Data_Status                                                             Interval_Note
2026-01-01               3           1         788869.17        708311.73        869426.61 Naive Forecast Synthetic demonstration only Approximate residual-based interval; not formally calibrated or validated
2026-02-01               3           2      

In [9]:

# Day 8 - Step 8: Forward-Looking Revenue Forecasts
# SYNTHETIC DEMONSTRATION ONLY

import numpy as np
import pandas as pd

# Load the complete 36-month synthetic dataset
revenue_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Synthetic_Monthly_Revenue.csv"
)

revenue_df = pd.read_csv(revenue_path, parse_dates=["Month"])
revenue_df = revenue_df.sort_values("Month").reset_index(drop=True)

# Use ALL available observations for the forward forecast
last_month = revenue_df["Month"].max()
last_revenue = float(revenue_df["Revenue"].iloc[-1])

# Estimate typical month-to-month changes from the full series
monthly_changes = revenue_df["Revenue"].diff().dropna()
change_std = float(monthly_changes.std(ddof=1))

print("Latest observed month:", last_month.strftime("%Y-%m"))
print("Latest observed synthetic revenue:", round(last_revenue, 2))
print("Historical monthly-change standard deviation:", round(change_std, 2))

# Create separate forward-looking forecast files
for horizon in [3, 6, 12]:
    rows = []

    for step in range(1, horizon + 1):
        forecast_month = last_month + pd.DateOffset(months=step)

        # Illustrative uncertainty range, not a calibrated prediction interval
        uncertainty = 1.96 * change_std * np.sqrt(step)

        rows.extend([
            {
                "Month": forecast_month,
                "Horizon_Months": horizon,
                "Step_Ahead": step,
                "Scenario": "Base",
                "Forecast_Revenue": round(last_revenue, 2),
                "Data_Status": "Synthetic demonstration only",
                "Assumption": "Revenue remains at the latest observed level"
            },
            {
                "Month": forecast_month,
                "Horizon_Months": horizon,
                "Step_Ahead": step,
                "Scenario": "Best-case",
                "Forecast_Revenue": round(last_revenue + uncertainty, 2),
                "Data_Status": "Synthetic demonstration only",
                "Assumption": "Illustrative upside scenario based on historical variation"
            },
            {
                "Month": forecast_month,
                "Horizon_Months": horizon,
                "Step_Ahead": step,
                "Scenario": "Worst-case",
                "Forecast_Revenue": round(max(0, last_revenue - uncertainty), 2),
                "Data_Status": "Synthetic demonstration only",
                "Assumption": "Illustrative downside scenario based on historical variation"
            }
        ])

    scenario_df = pd.DataFrame(rows)

    output_path = (
        project_dir / "data" / "synthetic"
        / f"Day8_Forward_Revenue_Scenarios_{horizon}M.csv"
    )

    scenario_df.to_csv(output_path, index=False)

    print(f"\n{horizon}-month scenarios saved to:")
    print(output_path)

    if horizon == 3:
        print("\n3-month scenario preview:")
        print(scenario_df.to_string(index=False))

print("\nStep 8 completed.")
print("These scenarios are illustrative, not calibrated forecasts.")


Latest observed month: 2025-12
Latest observed synthetic revenue: 788869.17
Historical monthly-change standard deviation: 41100.73

3-month scenarios saved to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Forward_Revenue_Scenarios_3M.csv

3-month scenario preview:
     Month  Horizon_Months  Step_Ahead   Scenario  Forecast_Revenue                  Data_Status                                                   Assumption
2026-01-01               3           1       Base         788869.17 Synthetic demonstration only                 Revenue remains at the latest observed level
2026-01-01               3           1  Best-case         869426.61 Synthetic demonstration only   Illustrative upside scenario based on historical variation
2026-01-01               3           1 Worst-case         708311.73 Synthetic demonstration only Illustrative downside scenario based on historical variation
2026-02-01               3           2       Base         788869.17 Synthetic demo

In [10]:

# Day 8 - Step 9: Revenue Forecasting Report
# SYNTHETIC DEMONSTRATION ONLY

from pathlib import Path
from datetime import datetime

report_path = project_dir / "reports" / "Day8_Revenue_Forecasting_Report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

report = """# Day 8 — Revenue Forecasting Report

## 1. Executive Summary

Day 8 established a revenue-data readiness audit, evaluated three forecasting
baselines on synthetic monthly data, and generated forward-looking scenario
files for 3, 6, and 12 months.

**Important:** All forecasting results in this report are synthetic
demonstrations. They are not actual NexziByte revenue, and they are not
validated business forecasts.

## 2. Revenue Data Readiness

- The keyword-based audit reviewed 24 CSV files.
- No potential revenue source was identified by the audit's filename and
  column-name matching rules.
- The prospect datasets do not contain verified revenue measures.
- The synthetic quotation dataset contains quote values, which must not be
  treated as realized revenue.
- Actual historical revenue data and business validation remain outstanding.

The audit is a screening exercise, not proof that revenue data does not exist
in other files or business systems.

## 3. Synthetic Dataset

- Frequency: Monthly
- Period: January 2023 through December 2025
- Number of observations: 36
- Latest synthetic observation: December 2025
- Latest synthetic revenue: 788,869.17
- Data status: Synthetic demonstration only

## 4. Backtesting Method

The first 30 months were used for training and the final 6 months for testing.
The final six months of the test period were July through December 2025.
Model metrics are therefore based on one six-month holdout and synthetic data.

## 5. Model Comparison

| Model | MAE | RMSE |
|---|---:|---:|
| Naive Forecast | 48,010.66 | 53,655.16 |
| Linear Trend | 53,156.25 | 59,105.72 |
| Seasonal Naive | 101,617.12 | 104,278.08 |

On this synthetic holdout, the naive forecast had the lowest MAE and RMSE.
This does not establish that it will be the best model on real business data.
A larger history and rolling-origin backtesting should be considered when
validated revenue records become available.

## 6. Forward-Looking Scenario Analysis

The forward-looking scenarios use all 36 synthetic observations, ending in
December 2025. The forecast periods begin in January 2026.

Scenario assumptions:
- Base: Revenue remains at the latest observed synthetic level.
- Best-case: An illustrative upside based on historical monthly variation.
- Worst-case: An illustrative downside based on historical monthly variation,
  bounded at zero.

The scenario ranges are illustrative and are not calibrated prediction
intervals, statistically validated confidence intervals, or business targets.

## 7. Key Limitations and Risks

1. No verified historical revenue dataset was identified by the audit.
2. All forecasting model comparisons use synthetic data.
3. Model ranking is based on a single six-month holdout.
4. The naive forecast assumes the latest training value remains constant.
5. Scenario assumptions are illustrative and do not establish causal drivers.
6. Seasonality and trend performance require validation on real data.
7. Business-owner confirmation of revenue definitions, granularity, and
   reporting periods is still required.

## 8. Deliverables

### Readiness and evaluation
- data/processed/Day8_Revenue_Data_Readiness_Audit.csv
- data/synthetic/Day8_Naive_Forecast_Backtest.csv
- data/synthetic/Day8_Naive_Forecast_Metrics.csv
- data/synthetic/Day8_Seasonal_Naive_Forecast_Backtest.csv
- data/synthetic/Day8_Linear_Trend_Forecast_Backtest.csv
- data/synthetic/Day8_Forecast_Model_Comparison.csv

### Synthetic dataset and forecasts
- data/synthetic/Day8_Synthetic_Monthly_Revenue.csv
- data/synthetic/Day8_Synthetic_Revenue_Forecast_3M.csv
- data/synthetic/Day8_Synthetic_Revenue_Forecast_6M.csv
- data/synthetic/Day8_Synthetic_Revenue_Forecast_12M.csv
- data/synthetic/Day8_Forward_Revenue_Scenarios_3M.csv
- data/synthetic/Day8_Forward_Revenue_Scenarios_6M.csv
- data/synthetic/Day8_Forward_Revenue_Scenarios_12M.csv

## 9. Conclusion and Next Actions

The technical forecasting demonstration is complete. Real-business forecasting
remains pending until actual revenue data and business-approved definitions
are available.

Next actions:
1. Obtain verified monthly revenue records from an approved source.
2. Confirm the revenue definition, currency, date field, and aggregation rules.
3. Re-run data-quality checks and backtesting on actual observations.
4. Compare models using rolling-origin evaluation.
5. Recalibrate prediction intervals and scenario assumptions.
6. Review findings with the business owner before operational use.

**Status:** Technical demonstration completed; real-business validation pending.
"""

report_path.write_text(report, encoding="utf-8")

print("Day 8 report saved to:")
print(report_path)
print("\nReport size:", report_path.stat().st_size, "bytes")
print("\nReport preview:")
print(report[:1200])


Day 8 report saved to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day8_Revenue_Forecasting_Report.md

Report size: 4688 bytes

Report preview:
# Day 8 — Revenue Forecasting Report

## 1. Executive Summary

Day 8 established a revenue-data readiness audit, evaluated three forecasting
baselines on synthetic monthly data, and generated forward-looking scenario
files for 3, 6, and 12 months.

**Important:** All forecasting results in this report are synthetic
demonstrations. They are not actual NexziByte revenue, and they are not
validated business forecasts.

## 2. Revenue Data Readiness

- The keyword-based audit reviewed 24 CSV files.
- No potential revenue source was identified by the audit's filename and
  column-name matching rules.
- The prospect datasets do not contain verified revenue measures.
- The synthetic quotation dataset contains quote values, which must not be
  treated as realized revenue.
- Actual historical revenue data and business validation remain outstand

In [11]:

# Day 8 - Step 10: Deliverable Verification

import pandas as pd

required_files = [
    "reports/Day8_Revenue_Forecasting_Report.md",
    "data/processed/Day8_Revenue_Data_Readiness_Audit.csv",
    "data/synthetic/Day8_Synthetic_Monthly_Revenue.csv",
    "data/synthetic/Day8_Naive_Forecast_Backtest.csv",
    "data/synthetic/Day8_Naive_Forecast_Metrics.csv",
    "data/synthetic/Day8_Seasonal_Naive_Forecast_Backtest.csv",
    "data/synthetic/Day8_Linear_Trend_Forecast_Backtest.csv",
    "data/synthetic/Day8_Forecast_Model_Comparison.csv",
    "data/synthetic/Day8_Synthetic_Revenue_Forecast_3M.csv",
    "data/synthetic/Day8_Synthetic_Revenue_Forecast_6M.csv",
    "data/synthetic/Day8_Synthetic_Revenue_Forecast_12M.csv",
    "data/synthetic/Day8_Forward_Revenue_Scenarios_3M.csv",
    "data/synthetic/Day8_Forward_Revenue_Scenarios_6M.csv",
    "data/synthetic/Day8_Forward_Revenue_Scenarios_12M.csv",
]

verification = []

for relative_path in required_files:
    path = project_dir / relative_path
    verification.append({
        "File": relative_path,
        "Exists": path.exists(),
        "Size_Bytes": path.stat().st_size if path.exists() else None
    })

verification_df = pd.DataFrame(verification)

print("DELIVERABLE VERIFICATION")
print(verification_df.to_string(index=False))

missing = verification_df.loc[~verification_df["Exists"], "File"].tolist()

print("\nMissing files:", len(missing))

# Check the forward scenario row counts
print("\nFORWARD SCENARIO ROW COUNTS")

for horizon in [3, 6, 12]:
    path = (
        project_dir / "data" / "synthetic"
        / f"Day8_Forward_Revenue_Scenarios_{horizon}M.csv"
    )

    if path.exists():
        df = pd.read_csv(path)
        print(
            f"{horizon}-month scenarios: {len(df)} rows "
            f"(expected {horizon * 3})"
        )

if not missing:
    print("\nAll expected Day 8 files exist.")
else:
    print("\nPlease investigate the missing files before sign-off.")


DELIVERABLE VERIFICATION
                                                    File  Exists  Size_Bytes
              reports/Day8_Revenue_Forecasting_Report.md    True        4688
    data/processed/Day8_Revenue_Data_Readiness_Audit.csv    True        1636
       data/synthetic/Day8_Synthetic_Monthly_Revenue.csv    True        2131
         data/synthetic/Day8_Naive_Forecast_Backtest.csv    True         229
          data/synthetic/Day8_Naive_Forecast_Metrics.csv    True         124
data/synthetic/Day8_Seasonal_Naive_Forecast_Backtest.csv    True         238
  data/synthetic/Day8_Linear_Trend_Forecast_Backtest.csv    True         236
       data/synthetic/Day8_Forecast_Model_Comparison.csv    True         289
   data/synthetic/Day8_Synthetic_Revenue_Forecast_3M.csv    True         550
   data/synthetic/Day8_Synthetic_Revenue_Forecast_6M.csv    True         997
  data/synthetic/Day8_Synthetic_Revenue_Forecast_12M.csv    True        1905
    data/synthetic/Day8_Forward_Revenue_Scenarios_3

In [12]:

# Day 8 - Step 8: Rolling-Origin Backtesting
# SYNTHETIC DEMONSTRATION ONLY — NOT ACTUAL NEXZIBYTE REVENUE

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the complete historical dataset.
revenue_path = (
    project_dir / "data" / "synthetic"
    / "Day8_Synthetic_Monthly_Revenue.csv"
)

rolling_df = pd.read_csv(revenue_path, parse_dates=["Month"])
rolling_df = rolling_df.sort_values("Month").reset_index(drop=True)

if rolling_df["Revenue"].isna().any():
    raise ValueError("Revenue contains missing values.")

if not rolling_df["Synthetic_Data_Flag"].astype(bool).all():
    raise ValueError("Expected all observations to be marked synthetic.")

values = rolling_df["Revenue"].to_numpy(dtype=float)
months = rolling_df["Month"]

# Use at least 24 months of history before evaluating a forecast.
# Evaluate the next 12 months one month at a time.
initial_train_size = 24
rolling_records = []

for origin in range(initial_train_size, len(values)):
    train_values = values[:origin]
    actual = float(values[origin])
    forecast_month = months.iloc[origin]

    # Model 1: Naive forecast
    naive_prediction = float(train_values[-1])

    # Model 2: Seasonal naive forecast (12-month seasonality)
    seasonal_prediction = float(
        train_values[-12]
    )

    # Model 3: Linear trend forecast
    X_train = np.arange(len(train_values)).reshape(-1, 1)
    X_next = np.array([[len(train_values)]])

    trend_model = LinearRegression()
    trend_model.fit(X_train, train_values)
    trend_prediction = float(trend_model.predict(X_next)[0])

    predictions = {
        "Naive Forecast": naive_prediction,
        "Seasonal Naive Forecast": seasonal_prediction,
        "Linear Trend Forecast": trend_prediction,
    }

    for model_name, prediction in predictions.items():
        rolling_records.append({
            "Forecast_Month": forecast_month,
            "Forecast_Origin_Month": months.iloc[origin - 1],
            "Training_Observations": origin,
            "Model": model_name,
            "Actual_Revenue": actual,
            "Forecast_Revenue": round(prediction, 2),
            "Absolute_Error": round(abs(actual - prediction), 2),
            "Squared_Error": round((actual - prediction) ** 2, 2),
            "Data_Status": "Synthetic demonstration only",
        })

rolling_results = pd.DataFrame(rolling_records)

# Summarize performance across all rolling forecast origins.
rolling_metrics = (
    rolling_results.groupby("Model")
    .apply(
        lambda group: pd.Series({
            "Forecast_Origins": group["Forecast_Month"].nunique(),
            "MAE": mean_absolute_error(
                group["Actual_Revenue"],
                group["Forecast_Revenue"]
            ),
            "RMSE": np.sqrt(mean_squared_error(
                group["Actual_Revenue"],
                group["Forecast_Revenue"]
            )),
        }),
        include_groups=False
    )
    .reset_index()
)

rolling_metrics["Data_Status"] = "Synthetic demonstration only"

# Save the detailed backtest and model comparison.
rolling_output = (
    project_dir / "data" / "synthetic"
    / "Day8_Rolling_Origin_Backtest.csv"
)

metrics_output = (
    project_dir / "data" / "synthetic"
    / "Day8_Rolling_Origin_Metrics.csv"
)

rolling_results.to_csv(rolling_output, index=False)
rolling_metrics.to_csv(metrics_output, index=False)

print("Rolling-origin backtest completed.")
print("Initial training observations:", initial_train_size)
print("Forecast origins:", rolling_results["Forecast_Month"].nunique())
print("Detailed rows:", len(rolling_results))

print("\nRolling-origin metrics:")
print(
    rolling_metrics.sort_values("MAE").to_string(index=False)
)

print("\nSaved detailed results:", rolling_output)
print("Saved summary metrics:", metrics_output)

print("\nAll results are synthetic demonstrations, not actual business forecasts.")

Rolling-origin backtest completed.
Initial training observations: 24
Forecast origins: 12
Detailed rows: 36

Rolling-origin metrics:
                  Model  Forecast_Origins           MAE          RMSE                  Data_Status
         Naive Forecast              12.0  31945.625833  40879.466257 Synthetic demonstration only
  Linear Trend Forecast              12.0  46405.072500  52485.061896 Synthetic demonstration only
Seasonal Naive Forecast              12.0 100682.283333 104139.274203 Synthetic demonstration only

Saved detailed results: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Rolling_Origin_Backtest.csv
Saved summary metrics: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day8_Rolling_Origin_Metrics.csv

All results are synthetic demonstrations, not actual business forecasts.
